In [7]:
import pandas as pd
import requests
import time

In [8]:
def classify_gpu_provider(row):
    provider = row["provider"]
    instance = str(row["instance_type"]).lower()

    if provider == "AWS":
        if instance.startswith(("p", "g", "inf", "trn")):
            return True

    if provider == "Azure":
        if instance.startswith("standard_n"):
            return True

    if provider == "GCP":
        if instance.startswith(("a2", "a3", "g2")) or "tpu" in instance:
            return True

    return False

In [9]:
pricing_df = pd.read_csv("normalized_vm_pricing.csv")

gpu_pricing_existing = pricing_df[
    pricing_df.apply(classify_gpu_provider, axis=1)
].copy()

gpu_pricing_existing.shape

(0, 11)

In [10]:
import pandas as pd
import requests

In [11]:
def fetch_azure_gpu_prices():

    rows = []

    url = "https://prices.azure.com/api/retail/prices"

    params = {
        "$filter":
        "serviceName eq 'Virtual Machines' "
        "and priceType eq 'Consumption'"
    }

    while url:

        r = requests.get(
            url,
            params=params
        )

        r.raise_for_status()

        data = r.json()

        for item in data.get("Items", []):

            sku = str(
                item.get(
                    "armSkuName",
                    ""
                )
            )

            if sku.lower().startswith(
                "standard_n"
            ):

                rows.append({

                    "provider":"Azure",

                    "region":
                    item.get(
                        "armRegionName"
                    ),

                    "instance_type":
                    sku,

                    "product_name":
                    item.get(
                        "productName"
                    ),

                    "hourly_price_usd":
                    item.get(
                        "retailPrice"
                    )

                })

        url = data.get(
            "NextPageLink"
        )

        params = None

    return pd.DataFrame(rows)

In [12]:
azure_gpu_df = fetch_azure_gpu_prices()

azure_gpu_df.shape

(6845, 5)

In [13]:
azure_gpu_df.head()

,provider,region,instance_type,product_name,hourly_price_usd
0,Azure,germanywestcentral,Standard_NC80adis_H100_v5,Virtual Machines NCadsH100v5 Series,3.630000
1,Azure,eastus2,Standard_NC320lds_xl_RTXPRO6000BSE_v6,Virtual Machines NCldsxlRTX6kv6,2.860000
2,Azure,eastus,Standard_NC320ds_xl_RTXPRO6000BSE_v6,Virtual Machines NCdsxlRTX6kv6 Windows,6.261600
3,Azure,switzerlandwest,Standard_ND96ams_A100_v4,Virtual Machines NDamsr A100 v4 Series Linux,15.838940
4,Azure,usgovarizona,Standard_NV4as_v4,Virtual Machines NVasv4 Series,0.053777


In [14]:
azure_gpu_df["instance_type"].nunique()

87

In [15]:
azure_gpu_df["instance_type"].value_counts().head(30)

instance_type
Standard_NV72ads_A10_v5      214
Standard_NV36ads_A10_v5      214
Standard_NV18ads_A10_v5      214
Standard_NV36adms_A10_v5     214
Standard_NV6ads_A10_v5       214
Standard_NV12ads_A10_v5      214
Standard_NC4as_T4_v3         192
Standard_NC8as_T4_v3         192
Standard_NC64as_T4_v3        188
Standard_NC16as_T4_v3        188
Standard_ND96ams_A100_v4     180
Standard_ND96amsr_A100_v4    174
Standard_NV16as_v4           158
Standard_NV32as_v4           158
Standard_NV4as_v4            158
Standard_NV8as_v4            158
Standard_NC24ads_A100_v4     156
Standard_NC48ads_A100_v4     156
Standard_NC96ads_A100_v4     156
Standard_NC40ads_H100_v5     146
Standard_NC80adis_H100_v5    146
Standard_NC24s_v3            146
Standard_NC24rs_v3           146
Standard_NC12s_v3            146
Standard_NC6s_v3             146
Standard_ND96isr_H100_v5     138
Standard_ND96isr_H200_v5     138
Standard_NV12s_v3            126
Standard_NV48s_v3            126
Standard_NV24s_v3            

In [16]:
azure_gpu_df.groupby(
    "instance_type"
)["hourly_price_usd"].mean().sort_values(
    ascending=False
).head(20)

instance_type
Standard_ND128isr_NDR_GB200_v6           143.380684
Standard_ND96isrf_H200_v5                108.208000
Standard_ND96isr_H200_v5                 103.310964
Standard_ND128isrf_NDR_GB200_v6           76.406400
Standard_ND96isrf_H100_v5                 72.043081
Standard_ND96isr_H100_v5                  65.870102
Standard_ND96isf_H100_v5                  65.522698
Standard_ND96is_flex_H100_v5              63.526789
Standard_ND96is_noIB_H100_v5              63.525422
Standard_ND96is_H100_v5                   57.173616
Standard_ND96isr_MI300X_v5                32.133292
Standard_ND96is_MI300X_v5                 32.133292
Standard_ND96ams_A100_v4                  24.728192
Standard_ND96amsr_A100_v4                 24.366569
Standard_ND96asr_v4                       17.399218
Standard_ND96asr_A100_v4                  17.039689
Standard_NC320ds_xl_RTXPRO6000BSE_v6      16.305000
Standard_NC320lds_xl_RTXPRO6000BSE_v6     14.665200
Standard_ND40rs_v2                        13.41022

In [17]:
def gpu_type(instance):

    text = str(instance).upper()

    if "GB200" in text:
        return "GB200"

    elif "H200" in text:
        return "H200"

    elif "H100" in text:
        return "H100"

    elif "A100" in text:
        return "A100"

    elif "MI300X" in text:
        return "MI300X"

    elif "A10" in text:
        return "A10"

    elif "RTXPRO6000" in text:
        return "RTX PRO 6000"

    else:
        return "Other"

In [18]:
azure_gpu_df["gpu_type"] = (
    azure_gpu_df["instance_type"]
    .apply(gpu_type)
)

In [19]:
azure_gpu_df["gpu_type"].value_counts()

gpu_type
Other           3187
A10             1338
A100             864
H100             714
RTX PRO 6000     316
MI300X           204
H200             142
GB200             80
Name: count, dtype: int64

In [20]:
azure_gpu_summary = (
    azure_gpu_df
    .groupby("gpu_type")
    .agg(
        vm_count=("instance_type","count"),
        unique_vm=("instance_type","nunique"),
        avg_price=("hourly_price_usd","mean")
    )
    .sort_values(
        "avg_price",
        ascending=False
    )
)

azure_gpu_summary

,vm_count,unique_vm,avg_price
gpu_type,,,
GB200,80,2,140.031970
H200,142,2,103.448908
H100,714,9,39.550567
MI300X,204,2,32.133292
A100,864,6,14.376439
RTX PRO 6000,316,12,6.789159
Other,3187,45,2.617441
A10,1338,9,2.248561


In [21]:
azure_gpu_df.to_csv(
    "azure_gpu_market_dataset.csv",
    index=False
)

azure_gpu_summary.to_csv(
    "azure_gpu_summary.csv"
)

In [22]:
azure_gpu_df["gpu_type"].value_counts()

gpu_type
Other           3187
A10             1338
A100             864
H100             714
RTX PRO 6000     316
MI300X           204
H200             142
GB200             80
Name: count, dtype: int64

In [23]:
azure_gpu_df[
    azure_gpu_df["gpu_type"]=="Other"
][
    "instance_type"
].value_counts().head(50)

instance_type
Standard_NC8as_T4_v3         192
Standard_NC4as_T4_v3         192
Standard_NC16as_T4_v3        188
Standard_NC64as_T4_v3        188
Standard_NV4as_v4            158
Standard_NV16as_v4           158
Standard_NV32as_v4           158
Standard_NV8as_v4            158
Standard_NC24rs_v3           146
Standard_NC12s_v3            146
Standard_NC6s_v3             146
Standard_NC24s_v3            146
Standard_NV24s_v3            126
Standard_NV48s_v3            126
Standard_NV12s_v3            126
Standard_NM16ads_MA35D        54
Standard_ND40rs_v2            45
Standard_NP20s                42
Standard_NP10s                42
Standard_NP40s                42
Standard_NG32ads_V620_v1      36
Standard_NV28adms_V710_v5     36
Standard_NV12ads_V710_v5      36
Standard_NV4ads_V710_v5       36
Standard_NG16ads_V620_v1      36
Standard_NV8ads_V710_v5       36
Standard_NV24ads_V710_v5      36
Standard_NG8ads_V620_v1       36
Standard_NG32adms_V620        36
Standard_ND40s_v2            

In [24]:
def gpu_type(instance):

    text = str(instance).upper()

    if "GB200" in text:
        return "GB200"

    elif "H200" in text:
        return "H200"

    elif "H100" in text:
        return "H100"

    elif "A100" in text:
        return "A100"

    elif "MI300X" in text:
        return "MI300X"

    elif "T4" in text:
        return "T4"

    elif "V100" in text:
        return "V100"

    elif "ND96ASR_V4" in text:
        return "A100"

    elif "ND40RS_V2" in text:
        return "V100"

    elif "NC6S_V3" in text:
        return "V100"

    elif "NC12S_V3" in text:
        return "V100"

    elif "NC24S_V3" in text:
        return "V100"

    elif "NC24RS_V3" in text:
        return "V100"

    elif "MA35D" in text:
        return "AMD MA35D"

    elif "V710" in text:
        return "AMD V710"

    elif "V620" in text:
        return "AMD V620"

    elif "_V4" in text and "NV" in text:
        return "AMD MI25"

    elif "A10" in text:
        return "A10"

    elif "RTXPRO6000" in text:
        return "RTX PRO 6000"

    else:
        return "Other"

In [25]:
azure_gpu_df["gpu_type"] = (
    azure_gpu_df["instance_type"]
    .apply(gpu_type)
)

azure_gpu_df["gpu_type"].value_counts()

gpu_type
A10             1338
A100             896
T4               760
Other            740
H100             714
AMD MI25         648
V100             629
RTX PRO 6000     316
MI300X           204
AMD V710         180
AMD V620         144
H200             142
GB200             80
AMD MA35D         54
Name: count, dtype: int64

In [26]:
def gpu_vendor(gpu):

    gpu = str(gpu)

    if gpu.startswith("AMD"):
        return "AMD"

    elif gpu in [
        "H100",
        "H200",
        "A100",
        "V100",
        "T4",
        "A10",
        "RTX PRO 6000",
        "GB200"
    ]:
        return "NVIDIA"

    else:
        return "Other"

In [27]:
azure_gpu_df["gpu_vendor"] = (
    azure_gpu_df["gpu_type"]
    .apply(gpu_vendor)
)

In [28]:
azure_gpu_df["gpu_vendor"].value_counts()

gpu_vendor
NVIDIA    4875
AMD       1026
Other      944
Name: count, dtype: int64

In [29]:
azure_gpu_df[
    azure_gpu_df["gpu_type"]=="Other"
]["instance_type"].value_counts().head(50)

instance_type
Standard_NV24s_v3       126
Standard_NV48s_v3       126
Standard_NV12s_v3       126
Standard_NP10s           42
Standard_NP20s           42
Standard_NP40s           42
Standard_ND40s_v2        33
Standard_NV24_Promo      28
Standard_NV12_Promo      28
Standard_NV6_Promo       28
Standard_NC24r_Promo     22
Standard_NC6_Promo       22
Standard_NC12_Promo      21
Standard_NC24_Promo      20
Standard_NV12s_v2        12
Standard_NV6s_v2         12
Standard_NV24s_v2        10
Name: count, dtype: int64

AWS GPU Data

In [30]:
import pandas as pd

In [31]:
aws_url = (
    "https://pricing.us-east-1.amazonaws.com/"
    "offers/v1.0/aws/AmazonEC2/current/index.csv"
)

print("Downloading AWS Price List...")

In [33]:
import pandas as pd

aws_url = (
    "https://pricing.us-east-1.amazonaws.com/"
    "offers/v1.0/aws/AmazonEC2/current/index.csv"
)

aws_raw = pd.read_csv(
    aws_url,
    skiprows=5,
    low_memory=False
)

aws_raw.shape

(7438769, 93)

In [34]:
aws_raw.columns.tolist()

['SKU',
 'OfferTermCode',
 'RateCode',
 'TermType',
 'PriceDescription',
 'EffectiveDate',
 'StartingRange',
 'EndingRange',
 'Unit',
 'PricePerUnit',
 'Currency',
 'RelatedTo',
 'LeaseContractLength',
 'PurchaseOption',
 'OfferingClass',
 'Product Family',
 'serviceCode',
 'Location',
 'Location Type',
 'Instance Type',
 'Current Generation',
 'Instance Family',
 'vCPU',
 'Physical Processor',
 'Clock Speed',
 'Memory',
 'Storage',
 'Network Performance',
 'Processor Architecture',
 'Storage Media',
 'Volume Type',
 'Max Volume Size',
 'Max IOPS/volume',
 'Max IOPS Burst Performance',
 'Max throughput/volume',
 'Provisioned',
 'Tenancy',
 'EBS Optimized',
 'Operating System',
 'License Model',
 'Group',
 'Group Description',
 'Transfer Type',
 'From Location',
 'From Location Type',
 'To Location',
 'To Location Type',
 'usageType',
 'operation',
 'AvailabilityZone',
 'CapacityStatus',
 'ClassicNetworkingSupport',
 'Dedicated EBS Throughput',
 'Dedicated EBS Throughput Description',
 

In [35]:
aws_raw["GPU"].value_counts().head(20)

GPU
1.000     177327
4.000      53426
8.000      33586
16.000      9091
2.000       6301
0.500       2600
0.125       2600
0.250       1300
12.000      1222
6.000       1222
Name: count, dtype: int64

In [36]:
aws_gpu = aws_raw[
    aws_raw["GPU"].notna()
].copy()

aws_gpu.shape

(288675, 93)

In [37]:
aws_gpu[
    [
        "Instance Type",
        "GPU",
        "GPU Memory"
    ]
].drop_duplicates().head(50)

,Instance Type,GPU,GPU Memory
25,g5.48xlarge,8.000,192 GB
131,inf1.2xlarge,1.000,NaN
143,g4dn.2xlarge,1.000,16 GB
176,g6.16xlarge,1.000,24 GB
255,g7e.48xlarge,8.000,768 GB
307,g6.2xlarge,1.000,24 GB
309,g6f.4xlarge,0.500,12 GB
409,p5.4xlarge,1.000,80 GB HBM3
483,trn1.2xlarge,1.000,32 GB
489,g5.16xlarge,1.000,24 GB


In [38]:
aws_gpu[
    "Instance Family"
].value_counts().head(20)

Instance Family
GPU instance                       251305
Machine Learning ASIC Instances     37370
Name: count, dtype: int64

In [39]:
aws_gpu_dataset = aws_gpu[
    [
        "SKU",
        "Location",
        "Region Code",
        "Instance Type",
        "Instance Family",
        "vCPU",
        "Memory",
        "GPU",
        "GPU Memory",
        "PricePerUnit",
        "Operating System"
    ]
].copy()

In [40]:
aws_gpu_dataset = aws_gpu_dataset[
    aws_gpu["Operating System"] == "Linux"
]

In [41]:
aws_gpu_dataset.shape

(83739, 11)

In [42]:
aws_gpu_dataset[
    [
        "Instance Type",
        "GPU",
        "GPU Memory",
        "PricePerUnit"
    ]
].sort_values(
    "PricePerUnit",
    ascending=False
).head(20)

,Instance Type,GPU,GPU Memory,PricePerUnit
3967621,g6.48xlarge,8.0,192 GB,2202647.0
3407358,g6.48xlarge,8.0,192 GB,2202647.0
4686240,g6.48xlarge,8.0,192 GB,2157072.0
259811,g6.48xlarge,8.0,192 GB,2157072.0
1711377,g6.48xlarge,8.0,192 GB,2138416.0
4283568,g6.48xlarge,8.0,192 GB,2138416.0
3407360,g6.48xlarge,8.0,192 GB,2130411.0
3967623,g6.48xlarge,8.0,192 GB,2130411.0
5019150,g6.48xlarge,8.0,192 GB,2129655.0
4460660,g6.48xlarge,8.0,192 GB,2129655.0


In [43]:
aws_raw["GPU"].value_counts().head(20)

GPU
1.000     177327
4.000      53426
8.000      33586
16.000      9091
2.000       6301
0.500       2600
0.125       2600
0.250       1300
12.000      1222
6.000       1222
Name: count, dtype: int64

In [44]:
aws_gpu.shape

(288675, 93)

In [45]:
aws_gpu["TermType"].value_counts()

TermType
Reserved    212990
OnDemand     75685
Name: count, dtype: int64

In [46]:
aws_gpu["Operating System"].value_counts().head(20)

Operating System
Linux                               83739
Windows                             70806
RHEL                                48776
Red Hat Enterprise Linux with HA    39437
SUSE                                39277
Ubuntu Pro                           6495
Name: count, dtype: int64

In [47]:
aws_gpu["Product Family"].value_counts()

Product Family
Compute Instance                 281037
Compute Instance (bare metal)      6487
Dedicated Host                     1151
Name: count, dtype: int64

In [48]:
aws_gpu["Instance Type"].value_counts().head(50)

Instance Type
g6.16xlarge      10620
g6.2xlarge       10620
g6.8xlarge       10620
g6.4xlarge       10620
g6.12xlarge      10620
g6.48xlarge      10620
g6.24xlarge      10620
g6.xlarge        10620
gr6.4xlarge       9720
gr6.8xlarge       9720
inf1.24xlarge     7985
inf1.xlarge       7852
inf1.2xlarge      7760
inf1.6xlarge      7622
g4dn.2xlarge      7590
g4dn.4xlarge      5937
g4dn.12xlarge     5622
g4dn.16xlarge     5558
g4dn.xlarge       5558
g4dn.8xlarge      5538
g4dn.metal        5095
g5.8xlarge        4580
g5.4xlarge        4580
g5.12xlarge       4530
g5.24xlarge       4530
g5.xlarge         4530
g5.2xlarge        4530
g5.48xlarge       4530
g5.16xlarge       4530
p5.4xlarge        3082
g6e.8xlarge       2750
g6e.48xlarge      2750
g6e.2xlarge       2750
g6e.16xlarge      2750
g6e.24xlarge      2750
g6e.xlarge        2750
g6e.4xlarge       2750
g6e.12xlarge      2750
p4d.24xlarge      2621
p5.48xlarge       2070
g4ad.8xlarge      1944
g4ad.xlarge       1944
g4ad.16xlarge     19

In [49]:
aws_gpu_clean = aws_gpu[
    (aws_gpu["TermType"] == "OnDemand") &
    (aws_gpu["Operating System"] == "Linux") &
    (aws_gpu["Product Family"] == "Compute Instance")
].copy()

In [50]:
aws_gpu_clean.shape

(16091, 93)

In [51]:
aws_gpu_clean[
    [
        "Instance Type",
        "GPU",
        "GPU Memory"
    ]
].drop_duplicates().head(100)

,Instance Type,GPU,GPU Memory
483,trn1.2xlarge,1.00,32 GB
489,g5.16xlarge,1.00,24 GB
821,g6.4xlarge,1.00,24 GB
1521,p2.16xlarge,16.00,NaN
1616,g6f.2xlarge,0.25,6 GB
...,...,...,...
393801,dl2q.24xlarge,8.00,128 GiB
428815,trn2.48xlarge,16.00,1536 GB HBM3
541662,g5g.8xlarge,1.00,16 GB
1200009,p6e-gb200.36xlarge,4.00,185 GB HBM3e


In [53]:
aws_gpu_clean = aws_gpu[
    (aws_gpu["TermType"] == "OnDemand") &
    (aws_gpu["Operating System"] == "Linux") &
    (aws_gpu["Product Family"] == "Compute Instance")
].copy()

aws_gpu_clean.shape

(16091, 93)

In [54]:
aws_gpu_clean[
    [
        "Instance Type",
        "GPU",
        "GPU Memory",
        "PricePerUnit"
    ]
].drop_duplicates().head(100)

,Instance Type,GPU,GPU Memory,PricePerUnit
483,trn1.2xlarge,1.00,32 GB,0.00000
489,g5.16xlarge,1.00,24 GB,0.00000
821,g6.4xlarge,1.00,24 GB,0.00000
1521,p2.16xlarge,16.00,NaN,0.00000
1616,g6f.2xlarge,0.25,6 GB,0.00000
...,...,...,...,...
49911,inf1.xlarge,1.00,NaN,1.74000
50453,g7e.2xlarge,1.00,96 GB,8.36527
50568,inf1.xlarge,1.00,NaN,0.70800
51940,g4dn.2xlarge,1.00,16 GB,1.07600


In [55]:
aws_gpu_clean["Instance Family"].value_counts().head(30)

Instance Family
GPU instance                       13257
Machine Learning ASIC Instances     2834
Name: count, dtype: int64

In [56]:
aws_gpu_clean["PricePerUnit"] = pd.to_numeric(
    aws_gpu_clean["PricePerUnit"],
    errors="coerce"
)

aws_gpu_clean = aws_gpu_clean[
    aws_gpu_clean["PricePerUnit"] > 0
].copy()

aws_gpu_clean.shape

(9409, 93)

In [57]:
aws_gpu_clean[
    [
        "Instance Type",
        "Instance Family",
        "GPU",
        "GPU Memory"
    ]
].drop_duplicates().sort_values(
    "Instance Type"
)

,Instance Type,Instance Family,GPU,GPU Memory
393801,dl2q.24xlarge,Machine Learning ASIC Instances,8.0,128 GiB
119083,g2.2xlarge,GPU instance,1.0,NaN
137922,g2.8xlarge,GPU instance,4.0,NaN
86165,g3.16xlarge,GPU instance,4.0,32 GB
139319,g3.4xlarge,GPU instance,1.0,8 GB
...,...,...,...,...
99168,p6-b200.48xlarge,GPU instance,8.0,1432 GB HBM3e
433838,p6-b300.48xlarge,GPU instance,8.0,2144 GB HBM3e
383330,trn1.2xlarge,Machine Learning ASIC Instances,1.0,32 GB
103495,trn1.32xlarge,Machine Learning ASIC Instances,16.0,512 GB


In [58]:
aws_gpu_clean["Instance Type"].nunique()

85

In [59]:
aws_family_summary = (
    aws_gpu_clean
    .groupby("Instance Family")
    .agg(
        vm_count=("Instance Type","nunique"),
        avg_gpu=("GPU","mean")
    )
)

aws_family_summary

,vm_count,avg_gpu
Instance Family,,
GPU instance,73,2.290935
Machine Learning ASIC Instances,12,5.624845


In [60]:
aws_gpu_clean["Instance Type"].nunique()

85

In [61]:
aws_gpu_clean[
    [
        "Instance Type",
        "Instance Family",
        "GPU",
        "GPU Memory"
    ]
].drop_duplicates().head(100)

,Instance Type,Instance Family,GPU,GPU Memory
2042,g7e.24xlarge,GPU instance,4.0,384 GB
2093,g6.48xlarge,GPU instance,8.0,192 GB
2131,inf1.xlarge,Machine Learning ASIC Instances,1.0,NaN
2235,inf1.2xlarge,Machine Learning ASIC Instances,1.0,NaN
3499,gr6.8xlarge,GPU instance,1.0,24 GB
...,...,...,...,...
536232,trn1n.32xlarge,Machine Learning ASIC Instances,16.0,512 GB
562417,g6e.16xlarge,GPU instance,1.0,48 GB
625038,g5g.8xlarge,GPU instance,1.0,16 GB
1094256,g5g.2xlarge,GPU instance,1.0,16 GB


In [62]:
aws_accelerator = (
    aws_gpu_clean[
        [
            "Instance Type",
            "Instance Family",
            "GPU",
            "GPU Memory"
        ]
    ]
    .drop_duplicates()
    .copy()
)

aws_accelerator.shape

(85, 4)

In [63]:
def accelerator_type(instance):

    text = str(instance).lower()

    if text.startswith("p5"):
        return "H100"

    elif text.startswith("p4"):
        return "A100"

    elif text.startswith("p3"):
        return "V100"

    elif text.startswith("g5"):
        return "A10G"

    elif text.startswith("g4"):
        return "T4"

    elif text.startswith("g6"):
        return "L4"

    elif text.startswith("g7"):
        return "Blackwell"

    elif text.startswith("trn1"):
        return "Trainium"

    elif text.startswith("trn2"):
        return "Trainium 2"

    elif text.startswith("inf1"):
        return "Inferentia"

    elif text.startswith("inf2"):
        return "Inferentia 2"

    else:
        return "Unknown"

In [64]:
aws_accelerator["accelerator"] = (
    aws_accelerator["Instance Type"]
    .apply(accelerator_type)
)

aws_accelerator["accelerator"].value_counts()

accelerator
L4              20
Unknown         15
A10G            13
T4              11
Blackwell        6
Inferentia       4
Inferentia 2     4
V100             4
H100             3
Trainium         3
A100             2
Name: count, dtype: int64

In [65]:
aws_accelerator.to_csv(
    "aws_gpu_market_dataset.csv",
    index=False
)

In [66]:
aws_accelerator["accelerator"].value_counts()

accelerator
L4              20
Unknown         15
A10G            13
T4              11
Blackwell        6
Inferentia       4
Inferentia 2     4
V100             4
H100             3
Trainium         3
A100             2
Name: count, dtype: int64

In [67]:
aws_accelerator[
    aws_accelerator["accelerator"]=="Unknown"
][
    [
        "Instance Type",
        "GPU",
        "GPU Memory"
    ]
].sort_values("Instance Type")

,Instance Type,GPU,GPU Memory
393801,dl2q.24xlarge,8.0,128 GiB
119083,g2.2xlarge,1.0,NaN
137922,g2.8xlarge,4.0,NaN
86165,g3.16xlarge,4.0,32 GB
139319,g3.4xlarge,1.0,8 GB
67154,g3.8xlarge,2.0,16 GB
125015,g3s.xlarge,1.0,8 GB
33834,gr6.4xlarge,1.0,24 GB
3499,gr6.8xlarge,1.0,24 GB
102999,gr6f.4xlarge,0.5,12 GB


In [68]:
def accelerator_type(instance):

    text = str(instance).lower()

    # 最新世代
    if text.startswith("p6-b300"):
        return "Blackwell Ultra (B300)"

    elif text.startswith("p6-b200"):
        return "Blackwell (B200)"

    elif text.startswith("p5"):
        return "H100"

    elif text.startswith("p4"):
        return "A100"

    elif text.startswith("p3"):
        return "V100"

    elif text.startswith("p2"):
        return "K80"

    elif text.startswith("g7"):
        return "Blackwell"

    elif text.startswith("g6"):
        return "L4"

    elif text.startswith("g5"):
        return "A10G"

    elif text.startswith("g4"):
        return "T4"

    elif text.startswith("g3"):
        return "M60"

    elif text.startswith("g2"):
        return "GRID K520"

    elif text.startswith("dl2q"):
        return "Gaudi 3"

    elif text.startswith("trn2"):
        return "Trainium 2"

    elif text.startswith("trn1"):
        return "Trainium"

    elif text.startswith("inf2"):
        return "Inferentia 2"

    elif text.startswith("inf1"):
        return "Inferentia"

    elif text.startswith("gr6"):
        return "NVIDIA L4"

    elif text.startswith("gr6f"):
        return "NVIDIA L4 Fractional"

    else:
        return "Unknown"

In [69]:
aws_accelerator["accelerator"] = (
    aws_accelerator["Instance Type"]
    .apply(accelerator_type)
)

aws_accelerator["accelerator"].value_counts()

accelerator
L4                        20
A10G                      13
T4                        11
Blackwell                  6
Inferentia                 4
Inferentia 2               4
V100                       4
M60                        4
NVIDIA L4                  3
H100                       3
K80                        3
Trainium                   3
A100                       2
GRID K520                  2
Blackwell (B200)           1
Gaudi 3                    1
Blackwell Ultra (B300)     1
Name: count, dtype: int64

In [70]:
aws_accelerator["accelerator"] = (
    aws_accelerator["accelerator"]
    .replace({
        "NVIDIA L4": "L4"
    })
)

In [71]:
aws_accelerator["accelerator"].value_counts()

accelerator
L4                        23
A10G                      13
T4                        11
Blackwell                  6
Inferentia                 4
Inferentia 2               4
M60                        4
V100                       4
H100                       3
K80                        3
Trainium                   3
A100                       2
GRID K520                  2
Blackwell (B200)           1
Gaudi 3                    1
Blackwell Ultra (B300)     1
Name: count, dtype: int64

In [72]:
aws_accelerator.to_csv(
    "aws_gpu_market_dataset.csv",
    index=False
)

GCP GPU Data

In [73]:
import pandas as pd
import requests
import time

In [74]:
GCP_API_KEY = "YOUR_GCP_API_KEY"
GCP_COMPUTE_SERVICE_ID = "6F81-5844-456A"

In [75]:
def fetch_gcp_accelerator_prices():
    rows = []

    url = f"https://cloudbilling.googleapis.com/v1/services/{GCP_COMPUTE_SERVICE_ID}/skus"
    params = {
        "key": GCP_API_KEY,
        "pageSize": 5000
    }

    while True:
        r = requests.get(url, params=params)
        r.raise_for_status()
        data = r.json()

        for sku in data.get("skus", []):
            desc = sku.get("description", "")
            text = desc.lower()

            is_accelerator = any(k in text for k in [
                "gpu",
                "nvidia",
                "a100",
                "h100",
                "h200",
                "l4",
                "t4",
                "v100",
                "p100",
                "tpu"
            ])

            if not is_accelerator:
                continue

            pricing_info = sku.get("pricingInfo", [])
            if not pricing_info:
                continue

            expr = pricing_info[0].get("pricingExpression", {})
            rates = expr.get("tieredRates", [])
            if not rates:
                continue

            unit_price = rates[0].get("unitPrice", {})
            price = (
                float(unit_price.get("units", 0)) +
                float(unit_price.get("nanos", 0)) / 1e9
            )

            rows.append({
                "provider": "GCP",
                "sku_id": sku.get("skuId"),
                "description": desc,
                "resource_family": sku.get("category", {}).get("resourceFamily"),
                "resource_group": sku.get("category", {}).get("resourceGroup"),
                "usage_type": sku.get("category", {}).get("usageType"),
                "regions": ",".join(sku.get("serviceRegions", [])),
                "hourly_price_usd": price,
                "currency": unit_price.get("currencyCode", "USD"),
                "source": "GCP Cloud Billing Catalog API"
            })

        token = data.get("nextPageToken")
        if not token:
            break

        params["pageToken"] = token
        time.sleep(0.2)

    return pd.DataFrame(rows)

In [76]:
gcp_accelerator_df = fetch_gcp_accelerator_prices()

gcp_accelerator_df.shape

(2586, 10)

In [77]:
gcp_accelerator_df.head(20)

,provider,sku_id,description,resource_family,resource_group,usage_type,regions,hourly_price_usd,currency,source
0,GCP,0008-F633-76AA,Nvidia L4 GPU attached to Spot Preemptible VMs...,Compute,GPU,Preemptible,asia-east2,0.212100,USD,GCP Cloud Billing Catalog API
1,GCP,0032-6F6D-C48E,Nvidia L4 GPU attached to Spot Preemptible VMs...,Compute,GPU,Preemptible,northamerica-northeast1,0.305800,USD,GCP Cloud Billing Catalog API
2,GCP,003E-D940-4BC0,Nvidia Tesla P100 GPU running in Seoul,Compute,GPU,OnDemand,asia-northeast3,1.600000,USD,GCP Cloud Billing Catalog API
3,GCP,0050-986A-2850,Commitment v1: H200 141GB GPU running in Nethe...,Compute,GPU,Commit1Yr,europe-west4,7.074530,USD,GCP Cloud Billing Catalog API
4,GCP,005F-DC8C-CB19,Nvidia H100 80GB Mega GPU running in Sydney,Compute,GPU,OnDemand,australia-southeast1,12.930340,USD,GCP Cloud Billing Catalog API
5,GCP,0071-242F-010B,Licensing Fee for Ubuntu Pro FIPS 20.04 LTS (F...,License,Canonical,OnDemand,global,0.035000,USD,GCP Cloud Billing Catalog API
6,GCP,008E-3414-1336,Commitment v1: Nvidia Tesla A100 GPU running i...,Compute,GPU,Commit3Yr,asia-east1,1.026868,USD,GCP Cloud Billing Catalog API
7,GCP,00B3-8705-5B5B,Reserved Nvidia H100 80GB Mega GPU in Los Ange...,Compute,GPU,OnDemand,us-west2,4.841121,USD,GCP Cloud Billing Catalog API
8,GCP,00E0-A1C1-4F05,Reserved Nvidia H100 80GB GPU in Frankfurt in ...,Compute,GPU,OnDemand,europe-west3,4.570091,USD,GCP Cloud Billing Catalog API
9,GCP,00F8-A9C1-00D0,Commitment v1: Nvidia L4 GPU running in Berlin...,Compute,GPU,Commit3Yr,europe-west10,0.388108,USD,GCP Cloud Billing Catalog API


In [78]:
gcp_accelerator_df["description"].value_counts().head(50)

description
Licensing Fee for Billing test license. (GPU cost)                                             4
Nvidia H100 80GB Mega GPU running in Sydney                                                    1
Licensing Fee for Ubuntu Pro FIPS 20.04 LTS (Focal Fossa) on VM with up to 1 GPU (GPU cost)    1
Commitment v1: Nvidia Tesla A100 GPU running in APAC for 3 Year                                1
Commitment v1: vGPU G4 Local SSD in Milan for 1 Year                                           1
Commitment v1: Nvidia Tesla P4 GPU running in Santiago for 1 Year                              1
Nvidia Tesla A100 80GB GPU running in Taiwan                                                   1
TpuV5e running in Dammam                                                                       1
Nvidia Tesla P100 GPU running in Seoul                                                         1
Commitment v1: Nvidia H100 80GB GPU running in Johannesburg for 3 Years                        1
Nvidia Tesla V100 

In [79]:
gcp_accelerator_clean = (
    gcp_accelerator_df[
        (
            gcp_accelerator_df["resource_group"]
            .isin(["GPU","TPU"])
        )
    ]
    .copy()
)

gcp_accelerator_clean.shape

(2394, 10)

In [80]:
def classify_gcp_accelerator(text):

    text = str(text).upper()

    if "B200" in text:
        return "Blackwell (B200)"

    elif "H200" in text:
        return "H200"

    elif "H100" in text:
        return "H100"

    elif "A100" in text:
        return "A100"

    elif "L4" in text:
        return "L4"

    elif "T4" in text:
        return "T4"

    elif "V100" in text:
        return "V100"

    elif "P100" in text:
        return "P100"

    elif "TPUV6E" in text:
        return "TPU v6e"

    elif "TPUV5P" in text:
        return "TPU v5p"

    elif "TPUV5E" in text:
        return "TPU v5e"

    elif "TPUV4" in text:
        return "TPU v4"

    else:
        return "Unknown"

In [81]:
gcp_accelerator_clean["accelerator"] = (
    gcp_accelerator_clean["description"]
    .apply(classify_gcp_accelerator)
)

In [82]:
gcp_accelerator_clean[
    "accelerator"
].value_counts()

accelerator
Unknown             438
A100                400
H100                310
TPU v5p             198
T4                  192
L4                  172
TPU v5e             156
Blackwell (B200)    136
P100                114
V100                108
H200                101
TPU v6e              69
Name: count, dtype: int64

In [83]:
gcp_accelerator_catalog = (
    gcp_accelerator_clean[
        [
            "accelerator",
            "resource_group"
        ]
    ]
    .drop_duplicates()
)

gcp_accelerator_catalog

,accelerator,resource_group
0,L4,GPU
2,P100,GPU
3,H200,GPU
4,H100,GPU
6,A100,GPU
10,TPU v5p,TPU
12,Blackwell (B200),GPU
15,T4,GPU
17,Unknown,GPU
26,TPU v6e,TPU


In [85]:
gcp_accelerator_clean[
    gcp_accelerator_clean["accelerator"]=="Unknown"
][
    [
        "description",
        "resource_group",
        "usage_type"
    ]
].drop_duplicates().head(100)

,description,resource_group,usage_type
17,Spot Preemptible 1/8 vGPU no lssd running in F...,GPU,Preemptible
22,Commitment v1: Nvidia Tesla P4 GPU running in ...,GPU,Commit1Yr
28,Nvidia Tesla P4 GPU attached to DWS Defined Du...,GPU,OnDemand
31,Commitment v1: vGPU G4 Local SSD in Frankfurt ...,GPU,Commit3Yr
47,Spot Preemptible vGPU G4 Instance Local SSD ru...,GPU,Preemptible
...,...,...,...
600,Spot Preemptible 1/8 vGPU no lssd running in P...,GPU,Preemptible
605,vGPU G4 Instance Local SSD running in Iowa,GPU,OnDemand
608,Spot Preemptible 1/8 vGPU no lssd running in M...,GPU,Preemptible
623,Reserved V5e TPU in Paris in Calendar Mode,TPU,OnDemand


In [86]:
gcp_accelerator_clean[
    gcp_accelerator_clean["accelerator"]=="Unknown"
]["description"].value_counts().head(50)

description
Spot Preemptible 1/8 vGPU no lssd running in Frankfurt                          1
Commitment v1: Nvidia Tesla P4 GPU running in Santiago for 1 Year               1
Nvidia Tesla P4 GPU attached to DWS Defined Duration VMs running in Americas    1
Commitment v1: vGPU G4 Local SSD in Frankfurt for 3 Year                        1
Spot Preemptible vGPU G4 Instance Local SSD running in Singapore                1
Commitment v1: vGPU G4 Local SSD in South Carolina for 1 Year                   1
Commitment v1: vGPU G4 Local SSD in Alabama for 3 Year                          1
Nvidia Tesla P4 GPU running in Seoul                                            1
1/8 vGPU no lssd attached to DWS Defined Duration VMs running in Dammam         1
Spot Preemptible 1/8 vGPU no lssd running in Netherlands                        1
Commitment v1: vGPU G4 Local SSD in Frankfurt for 1 Year                        1
Nvidia Tesla P4 GPU running in Turin                                            1
Nvid

In [88]:
def classify_gcp_accelerator(text):

    text = str(text).upper()

    if "A100" in text:
        return "A100"

    elif "H100" in text:
        return "H100"

    elif "H200" in text:
        return "H200"

    elif "P100" in text:
        return "P100"

    elif "V100" in text:
        return "V100"

    elif "L4" in text:
        return "L4"

    elif "T4" in text:
        return "T4"

    elif "P4" in text:
        return "P4"

    elif "VGPU G4" in text:
        return "T4"

    elif "1/8 VGPU" in text:
        return "T4 Fractional"

    elif "TPU7X" in text:
        return "TPU v7"

    elif "V5E TPU" in text:
        return "TPU v5e"

    elif "V6E TPU" in text:
        return "TPU v6e"

    else:
        return "Unknown"

In [89]:
gcp_accelerator_clean["accelerator"] = (
    gcp_accelerator_clean["description"]
    .apply(classify_gcp_accelerator)
)

gcp_accelerator_clean["accelerator"].value_counts()

accelerator
Unknown          559
A100             400
T4               317
H100             310
L4               172
T4 Fractional    125
P4               124
P100             114
V100             108
H200             101
TPU v5e           42
TPU v6e           12
TPU v7            10
Name: count, dtype: int64

In [90]:
def classify_gcp_accelerator(text):

    text = str(text).upper()

    if "B200" in text:
        return "Blackwell (B200)"

    elif "H200" in text:
        return "H200"

    elif "H100" in text:
        return "H100"

    elif "A100" in text:
        return "A100"

    elif "L4" in text:
        return "L4"

    elif "T4" in text:
        return "T4"

    elif "P4" in text:
        return "P4"

    elif "P100" in text:
        return "P100"

    elif "V100" in text:
        return "V100"

    elif "VGPU G4" in text:
        return "T4"

    elif "1/8 VGPU" in text:
        return "T4 Fractional"

    elif "TPU7X" in text:
        return "TPU v7"

    elif "TPUV6E" in text or "V6E TPU" in text:
        return "TPU v6e"

    elif "TPUV5P" in text or "V5P TPU" in text:
        return "TPU v5p"

    elif "TPUV5E" in text or "V5E TPU" in text:
        return "TPU v5e"

    elif "TPUV4" in text or "V4 TPU" in text:
        return "TPU v4"

    else:
        return "Unknown"

In [91]:
gcp_accelerator_clean["accelerator"] = (
    gcp_accelerator_clean["description"]
    .apply(classify_gcp_accelerator)
)

gcp_accelerator_clean["accelerator"].value_counts()

accelerator
A100                400
T4                  317
H100                310
TPU v5p             198
TPU v5e             198
L4                  172
Blackwell (B200)    136
T4 Fractional       125
P4                  124
P100                114
V100                108
H200                101
TPU v6e              81
TPU v7               10
Name: count, dtype: int64

In [92]:
gcp_accelerator_clean.to_csv(
    "gcp_accelerator_dataset.csv",
    index=False
)

In [93]:
gcp_accelerator_summary = (
    gcp_accelerator_clean
    .groupby("accelerator")
    .agg(
        sku_count=("sku_id", "count"),
        avg_price=("hourly_price_usd", "mean"),
        median_price=("hourly_price_usd", "median")
    )
    .reset_index()
    .sort_values("sku_count", ascending=False)
)

gcp_accelerator_summary.to_csv(
    "gcp_accelerator_summary.csv",
    index=False
)

gcp_accelerator_summary

,accelerator,sku_count,avg_price,median_price
0,A100,400,2.199114,1.869319
7,T4,317,0.170112,0.160000
2,H100,310,6.158232,4.841121
9,TPU v5e,198,0.886567,0.840000
10,TPU v5p,198,2.976808,2.940000
4,L4,172,0.426978,0.397387
1,Blackwell (B200),136,8.057794,8.055000
8,T4 Fractional,125,0.440452,0.336380
6,P4,124,0.415742,0.409000
5,P100,114,1.015138,1.008000
